# Выбор и группировка в pandas

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Выбирать строки и столбцы, считать групповые показатели и учитывать пропуски.

## DataFrame и Series

`DataFrame` — таблица с именованными столбцами и индексом строк. Один столбец — `Series`. Индекс — метка строки, он не обязан совпадать с её позицией или быть уникальным. Создадим вымышленные измерения четырёх планеров. Подключим библиотеку pandas под именем `pd`. `pd.DataFrame` строит таблицу из словаря: ключи задают столбцы, списки — их значения. Свойство `shape` задаёт число строк и столбцов.

`None` в числовом столбце становится пропуском; его конкретное представление зависит от типа столбца. `frame["mass_g"]` выбирает один столбец; его `isna()` даёт маску пропусков, а `sum()` считает значения `True` как единицы. Свойство `dtypes` показывает типы столбцов. Проверяйте типы (`dtypes`), первые строки (`head`) и сводку (`describe`) перед расчётами. Для чтения своего файла используйте `pd.read_csv(path)`; разделитель и кодировку задавайте по формату файла.

In [ ]:
import pandas as pd

frame = pd.DataFrame({
    "id": ["a", "b", "c", "d"],
    "span_cm": [12., 22., 14., 24.],
    "mass_g": [8., 10., None, 12.],
    "flight": ["короткий", "дальний", "короткий", "дальний"],
})
print(frame.shape)              # (4, 4)
print(frame["mass_g"].isna().sum())  # 1
print(frame.dtypes)


## Выбор и группировка

`.loc` выбирает по меткам или логической маске, `.iloc` — по позициям. `frame.loc[маска, список_столбцов]` оставляет нужные строки и столбцы, `frame.iloc[0]` возвращает первую строку. Метод `tolist()` превращает столбец в обычный список, `to_dict()` — результат с метками в словарь. `groupby` делит строки на группы перед вычислением показателя; `mean()` считает среднее, `count()` — число заполненных значений, `size()` — число всех строк группы. Для изменения используйте одно обращение `frame.loc[mask, "column"] = value`, а не цепочку промежуточных выборок. Если нужна независимая рабочая таблица, сделайте `.copy()`.

In [ ]:
selected = frame.loc[frame["span_cm"] >= 20, ["id", "span_cm"]]
assert selected["id"].tolist() == ["b", "d"]
assert frame.iloc[0]["id"] == "a"
means = frame.groupby("flight")["span_cm"].mean()
assert means["короткий"] == 13
assert means["дальний"] == 23


Среднее по умолчанию пропускает отсутствующие значения. Поэтому вместе с ним полезно считать количество заполненных (`count`) и всех строк (`size`). Две группы с одним и со ста измерениями дают оценки разной надёжности.

## Разобранный пример: сколько данных стоит за средним

Среднее само по себе не показывает число наблюдений. В нашей таблице в группе «короткий» два объекта, но заполненная масса только у одного. Сравним число всех строк и заполненных измерений.

In [ ]:
row_counts = frame.groupby("flight").size()
measured_counts = frame.groupby("flight")["mass_g"].count()
assert row_counts["короткий"] == 2
assert measured_counts["короткий"] == 1


Метод `size` относится к группам строк, `count` — к заполненным значениям выбранного столбца. Для одного и того же набора они могут различаться. При расчёте среднего укажите, какие пропуски исключены и сколько измерений осталось.

## Индекс и идентификатор

Индекс pandas — метка строки. Наличие столбца `id` не делает его автоматически индексом. Позиция 0 обозначает первую строку текущего порядка, даже если её индекс равен 10. Поэтому `.iloc` удобно для позиционного примера, `.loc` — для выбора по меткам и условиям.

## Проверяем разницу метки и позиции

Поменяем только метки строк в копии. Число 10 станет именем первой строки, но её позиция останется 0.

In [ ]:
labeled = frame.copy()
labeled.index = [10, 20, 30, 40]
assert labeled.loc[10, "id"] == "a"
assert labeled.iloc[0]["id"] == "a"


Вызов `labeled.loc[0]` не найдёт метку и вызовет `KeyError`. Это не признак пустой таблицы: строки есть, но ни одна не имеет метки 0. Маска `frame["span_cm"] >= 20` содержит `[False, True, False, True]`, поэтому выбираются именно объекты b и d.

## Задачи

### 1. Базовая

Чем отличаются loc и iloc?

<details>
<summary>Подсказка</summary>

Метка и позиция могут различаться.

</details>



### 2. Базовая

Сколько заполненных масс в исходной таблице?

<details>
<summary>Подсказка</summary>

У одного из четырёх объектов пропуск.

</details>



### 3. Прикладная

Посчитайте число объектов каждого класса.

<details>
<summary>Подсказка</summary>

groupby и size.

</details>



### 4. Прикладная

Найдите id объектов с размахом больше 20.

<details>
<summary>Подсказка</summary>

Выберите строки и один столбец.

</details>



### 5. Олимпиадная

Почему одинаковые средние не означают одинаковую надёжность оценки?

<details>
<summary>Подсказка</summary>

Под ними могут быть разные количества и разбросы.

</details>



### 6. Олимпиадная

Если count равен 0, можно ли считать среднюю массу нулевой?

<details>
<summary>Подсказка</summary>

Измерений нет.

</details>



## Итоги

Вы умеете выбирать строки и столбцы, группировать данные и учитывать число заполненных значений.

## Самопроверка

Объясните разницу между индексом и позицией строки. Когда `count` и `size` дадут разные ответы? Вручную проверьте средний размах одной группы.

Для перехода дальше достаточно уверенно решать базовые и прикладные задачи. К олимпиадным можно вернуться позже.


## Решения задач

### Задача 1

loc выбирает по меткам или маскам, iloc — по позициям.

### Задача 2

Три.

### Задача 3

In [ ]:
assert frame.groupby("flight").size().to_dict() == {"дальний": 2, "короткий": 2}


### Задача 4

In [ ]:
assert frame.loc[frame["span_cm"] > 20, "id"].tolist() == ["b", "d"]


### Задача 5

Одно среднее может опираться на один объект, другое — на сотню. Нужны объём данных и распределение значений.

### Задача 6

Нет. Ноль — возможное измерение, а отсутствие среднего означает недостаток данных.